In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib scikit-learn tqdm
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q librosa soundfile tensorboard
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU不可用，请在运行时类型中选择T4 GPU')

# 从感知机到神经网络

## 学习目标

- 理解"学习"就是"调参数"的过程
- 用纯Python实现线性回归，观察梯度下降
- 用PyTorch重写同样的模型，理解框架帮我们做了什么
- 理解损失函数、梯度、学习率的概念

## 本节核心思想

> **深度学习 = 用数据自动调整参数**。仅此而已。
> 所谓的"学习"，就是不断微调参数，让预测结果越来越接近真实值。


## 1. 线性回归：最简单的"学习"

假设我们有一些数据点，想找到一条直线来拟合它们。

这就是最简单的"学习"：找到最好的参数(w, b)，让 $y = wx + b$ 尽可能接近真实数据。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 4)

### 1.1 生成数据

我们生成一组"频率 → 振幅衰减"的数据：频率越高，振幅衰减越大。
这在CI研究中很常见——高频通道的增益通常低于低频通道。

In [ ]:
# 生成模拟数据：频率 → 振幅衰减
freq = np.random.uniform(100, 8000, 50)  # 频率 100-8000 Hz
true_w = -0.00005  # 真实斜率：频率越高，振幅越小
true_b = 0.9       # 真实截距
amplitude = true_w * freq + true_b + np.random.normal(0, 0.05, 50)  # 加噪声

plt.figure(figsize=(10, 4))
plt.scatter(freq, amplitude, alpha=0.6)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.title('Frequency vs Amplitude Attenuation')
plt.grid(True, alpha=0.3)
plt.show()

### 1.2 纯Python实现梯度下降

现在我们要"学习"到 w 和 b 的值。

**核心流程**：
1. 随机初始化 w 和 b
2. 用当前的 w 和 b 计算预测值
3. 计算预测值和真实值的差距（损失）
4. 计算"往哪个方向调参数能让损失变小"（梯度）
5. 调整参数（梯度下降）
6. 重复 2-5

> **重要**：在训练前，需要对输入特征做归一化。因为 `freq` 的值在 100–8000 之间，
> 导致 `dw`（梯度对 w）比 `db` 大几千倍，使得同一个学习率无法同时让 w 和 b 收敛。
> 归一化后，特征的尺度相近，梯度也更均衡，学习率的选择变得容易。

In [ ]:
# ====== 纯Python实现梯度下降 ======

# 关键步骤：归一化输入特征！
# freq 的范围是 100~8000，直接使用会导致梯度严重不平衡：
#   dw ≈ (2/n) * sum(error * freq)   —— 因为 freq 很大，dw 很大
#   db ≈ (2/n) * sum(error)           —— 没有 freq 放大，db 很小
# 同一个学习率无法同时让 w 和 b 收敛。归一化后二者梯度量级接近。
freq_mean = freq.mean()
freq_std = freq.std()
freq_norm = (freq - freq_mean) / freq_std  # 归一化后均值0、标准差1

# 初始化参数（随机值）
w = 0.0
b = 0.0
learning_rate = 0.01    # 归一化后可以用合理的学习率
n_epochs = 300          # 训练轮数
n_samples = len(freq_norm)

# 记录训练过程
losses = []
ws = []
bs = []

for epoch in range(n_epochs):
    # 前向传播：用归一化后的特征计算预测值
    predictions = w * freq_norm + b
    
    # 计算损失（均方误差）
    loss = np.mean((predictions - amplitude) ** 2)
    losses.append(loss)
    
    # 计算梯度（损失对参数的偏导数）
    dw = (2 / n_samples) * np.sum((predictions - amplitude) * freq_norm)
    db = (2 / n_samples) * np.sum(predictions - amplitude)
    
    # 更新参数
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    ws.append(w)
    bs.append(b)

# 还原到原始尺度的参数
# 原始模型: amplitude = w_orig * freq + b_orig
# 归一化模型: amplitude = w * (freq - mean) / std + b
#            = (w / std) * freq + (b - w * mean / std)
w_orig = w / freq_std
b_orig = b - w * freq_mean / freq_std

print(f'训练结果: w = {w_orig:.8f}, b = {b_orig:.4f}')
print(f'真实值:   w = {true_w:.8f}, b = {true_b:.4f}')
print(f'最终损失: {losses[-1]:.6f}')

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 损失曲线
axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# 参数w的变化（原始尺度）
ws_orig = [wi / freq_std for wi in ws]
axes[1].plot(ws_orig, label='Learned w')
axes[1].axhline(y=true_w, color='r', linestyle='--', label='True w')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('w')
axes[1].set_title('Parameter w (original scale)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 拟合结果（用原始尺度的参数）
x_line = np.linspace(100, 8000, 100)
axes[2].scatter(freq, amplitude, alpha=0.4, label='Data')
axes[2].plot(x_line, w_orig * x_line + b_orig, 'r-', linewidth=2, label='Fitted line')
axes[2].plot(x_line, true_w * x_line + true_b, 'g--', linewidth=2, label='True line')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('Amplitude')
axes[2].set_title('Fitted Result')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.3 关键概念

**损失函数 (Loss Function)**：衡量"预测值和真实值差多远"。上面用的是均方误差(MSE)。

**梯度 (Gradient)**：告诉参数"该变大还是变小"。梯度就是损失函数对参数的偏导数。

**学习率 (Learning Rate)**：每次调参数的步长。太大→震荡，太小→慢。

**梯度下降 (Gradient Descent)**：沿着梯度的反方向调整参数。就像下山——沿着最陡的方向往下走。

### 练习1：调整学习率

修改上面的代码，尝试不同的学习率，观察训练过程：
- `learning_rate = 0.001`（较小）
- `learning_rate = 0.01`（合适）
- `learning_rate = 0.1`（较大）
- `learning_rate = 1.0`（太大，会发散）

**问题**：学习率太大时，损失曲线会怎样？为什么？

**延伸思考**：如果去掉归一化（直接用原始的 `freq`），上面的学习率还能用吗？试试看会发生什么。

## 2. 用PyTorch重写——框架帮我们做了什么？

刚才我们手动实现了梯度计算和参数更新。PyTorch帮我们自动做了这两件事：
1. **自动微分**：不需要手算梯度
2. **优化器**：不需要手写参数更新规则

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 准备数据（转为PyTorch张量）—— 同样需要归一化！
freq_norm_tensor = torch.FloatTensor(freq_norm).unsqueeze(1)   # shape: [50, 1]
amp_tensor = torch.FloatTensor(amplitude).unsqueeze(1)         # shape: [50, 1]

# 定义模型
model = nn.Linear(1, 1)  # 输入1维，输出1维

# 定义损失函数和优化器（归一化后学习率可以合理设置）
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 训练
losses_torch = []
for epoch in range(300):
    # 前向传播
    predictions = model(freq_norm_tensor)
    
    # 计算损失
    loss = criterion(predictions, amp_tensor)
    losses_torch.append(loss.item())
    
    # 反向传播（自动计算梯度！）
    optimizer.zero_grad()  # 清零梯度
    loss.backward()        # 计算梯度
    optimizer.step()       # 更新参数

# 还原到原始尺度的参数
w_learned_norm = model.weight.item()
b_learned_norm = model.bias.item()
w_learned = w_learned_norm / freq_std
b_learned = b_learned_norm - w_learned_norm * freq_mean / freq_std

print(f'PyTorch结果: w = {w_learned:.8f}, b = {b_learned:.4f}')
print(f'纯Python结果: w = {w_orig:.8f}, b = {b_orig:.4f}')
print(f'真实值:       w = {true_w:.8f}, b = {true_b:.4f}')

### 对比

| 方面 | 纯Python | PyTorch |
|------|---------|----------|
| 梯度计算 | 手动推导偏导数 | `loss.backward()` 自动计算 |
| 参数更新 | 手动写 `w = w - lr * dw` | `optimizer.step()` 自动更新 |
| 模型定义 | 直接用变量 | `nn.Linear` 封装 |
| 代码量 | 多 | 少 |

**PyTorch帮我们省略了最繁琐的部分（梯度计算和参数更新），让我们专注于模型设计。**

## 3. 张量(Tensor)——PyTorch的核心数据结构

张量就是"多维数组"，和NumPy的ndarray很像，但可以在GPU上运算。

In [ ]:
# 张量基础
import torch

# 创建张量
a = torch.tensor([1.0, 2.0, 3.0])       # 从列表创建
b = torch.zeros(3)                       # 全0
c = torch.ones(2, 3)                     # 全1，形状2x3
d = torch.randn(2, 3)                    # 随机数，形状2x3

print(f'a = {a}')
print(f'b = {b}')
print(f'c = {c}')
print(f'd = {d}')
print(f'd.shape = {d.shape}')  # 形状

In [ ]:
# 张量运算
x = torch.randn(3, 4)
y = torch.randn(3, 4)

print('逐元素运算:')
print(f'x + y 的形状: {(x + y).shape}')
print(f'x * y 的形状: {(x * y).shape}')  # 逐元素乘法

print('\n矩阵运算:')
z = torch.randn(4, 2)
print(f'x @ z 的形状: {(x @ z).shape}')  # 矩阵乘法

print('\n与NumPy互转:')
np_array = x.numpy()           # Tensor → NumPy
tensor_from_np = torch.from_numpy(np_array)  # NumPy → Tensor
print(f'互转后数据一致: {np.allclose(x.numpy(), tensor_from_np.numpy())}')

In [ ]:
# GPU加速
if torch.cuda.is_available():
    x_gpu = x.to('cuda')
    print(f'张量在设备: {x_gpu.device}')
    print(f'GPU矩阵乘法...')
    
    # 大矩阵乘法对比
    import time
    size = 2000
    
    # CPU
    a_cpu = torch.randn(size, size)
    b_cpu = torch.randn(size, size)
    start = time.time()
    c_cpu = a_cpu @ b_cpu
    cpu_time = time.time() - start
    
    # GPU
    a_gpu = a_cpu.to('cuda')
    b_gpu = b_cpu.to('cuda')
    start = time.time()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    
    print(f'CPU: {cpu_time*1000:.1f} ms, GPU: {gpu_time*1000:.1f} ms')
    print(f'加速比: {cpu_time/gpu_time:.1f}x')
else:
    print('GPU不可用，跳过GPU演示')

## 4. 自动微分——PyTorch最强大的功能

PyTorch的 `autograd` 系统可以自动计算梯度，你不需要手算任何偏导数。

In [ ]:
# 自动微分示例
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 做一些运算
y = x ** 2       # y = x²
z = y.sum()      # z = Σx²

# 反向传播——自动计算梯度
z.backward()

# 查看梯度：dz/dx = 2x
print(f'x = {x}')
print(f'x的梯度 (应为2x) = {x.grad}')
print(f'验证: 2 * x = {2 * x}')

### 自动微分的意义

上面的例子中，$z = \sum x_i^2$，手动计算 $\frac{\partial z}{\partial x_i} = 2x_i$。

PyTorch帮我们自动完成了这个计算。对于简单的函数，手算不难；
但对于深度神经网络（几十层，上百万参数），手动计算梯度几乎不可能——这就是自动微分的价值。

**记住**：`loss.backward()` = 自动计算所有参数的梯度；`optimizer.step()` = 用梯度更新参数。你只需要做这两步。

## 本节要点

1. **学习 = 调参数**：找到最好的参数让预测接近真实值
2. **梯度下降**：沿梯度的反方向调整参数，就像下山
3. **损失函数**：衡量预测和真实的差距
4. **学习率**：步长，太大→震荡，太小→慢
5. **特征归一化**：训练前必须归一化输入特征，否则梯度量级差异巨大，参数无法同时收敛
6. **PyTorch做了什么**：自动微分(gradient) + 优化器(optimizer)，让我们专注模型设计
7. **张量**：PyTorch的核心数据结构，和NumPy数组类似但支持GPU

---
下一节：[02-mlp-pytorch.ipynb](02-mlp-pytorch.ipynb) — MLP与反向传播